In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Set, Dict
from collections import defaultdict
import os

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [2]:
bace_df = pd.read_csv("bace.csv")
cvae_df = pd.read_csv("vae/paper_based/CVAE/10k_bace_test.txt.csv")

In [ ]:
zinc_dictionary = {}
cvae_dictionary = {}


for i, row in bace_df.iterrows():
    mol = Chem.MolFromSmiles(row["mol"])

    lfg = rdMolStandardize.LargestFragmentChooser()
    mol = lfg.choose(mol)

    uncharger = rdMolStandardize.Uncharger()
    mol = uncharger.uncharge(mol)

    smiles = Chem.MolToSmiles(mol)

    smiles_value = zinc_dictionary.get(smiles)
    if smiles_value == None:
        zinc_dictionary[smiles] = row["pIC50"]


for i, row in cvae_df.iterrows():
    mol = Chem.MolFromSmiles(row["smiles"])

    lfg = rdMolStandardize.LargestFragmentChooser()
    mol = lfg.choose(mol)

    uncharger = rdMolStandardize.Uncharger()
    mol = uncharger.uncharge(mol)

    smiles = Chem.MolToSmiles(mol)

    
    if cvae_dictionary.get(smiles) == None and cvae_dictionary.get(smiles) == None:
        cvae_dictionary[smiles] = row["pred_pIC50"]

In [4]:
processed_data_setups = [(10000, 1), (10000, 0.9), (5000, 0.8), (3000, 0.7),
                         (1000, 0), (1000, 0.5), (1000, 1), 
                         (500, 0), (500, 0.5), (1000, 1),
                         (250, 0), (250, 0.5), (250, 1)] # (Mol Count, Synthetic Fraction)

processed_data_combinations = []

for count, synthetic in processed_data_setups:
    combination = {}
    i = 0
    for smiles, value in zinc_dictionary.items():
        combination[smiles] = value
        i += 1
        if i >= round(count * (1-synthetic)): break
    for smiles, value in cvae_dictionary.items():
        combination[smiles] = value
        i += 1
        if i >= round(count * synthetic): break
    processed_data_combinations.append(combination)

In [5]:
scaffold_datasets = []

for combination in processed_data_combinations:
    dataset = defaultdict(list)
    for smiles, logP in combination.items():
        mol = Chem.MolFromSmiles(smiles)

        try: # From what I've understood, there are atoms with more than 4 bonds in the dataset, catch these errors and skip them.
            scaffold = MurckoScaffold.MakeScaffoldGeneric(mol)
        except Chem.AtomValenceException:
            continue

        scaffold_smiles = Chem.MolToSmiles(scaffold, canonical=True)

        if scaffold_smiles != '' and scaffold.GetNumAtoms() > 0:
                dataset[scaffold_smiles].append(smiles)
    scaffold_datasets.append(dataset)

In [7]:
for combination_index in range(len(processed_data_setups)):
    fold_size = processed_data_setups[combination_index][0] / 5

    for fold_iteration_index in range(5):
        fold_iteration_data = []
        fold_counts = [fold_iteration_index * fold_size/5, 0, 0, 0, 0]

        for scaffold, associated_smiles in scaffold_datasets[combination_index].items():
            fold_index = 0
            for i in range(6):
                fold_index = 6 % 5
                if fold_counts[fold_index] + len(associated_smiles) > fold_size:
                    break
            for smiles in associated_smiles:
                fold_iteration_data.append((fold_index, smiles, processed_data_combinations[combination_index][smiles]))
            fold_counts[fold_index] += 1

        directory = f"combination_{processed_data_setups[combination_index][0]}_molecules_and_{int(processed_data_setups[combination_index][1]*100)}_%_synthetic"
        if not os.path.exists(directory): os.makedirs(directory)
        fold_df = pd.DataFrame(data=fold_iteration_data, columns=["fold", "smiles", "pIC50"])
        fold_df.to_csv(f"{directory}/fold_iteration_{fold_iteration_index}.csv")